In [41]:
import pandas as pd
import functools

from meridian.model import model
from meridian import constants

In [2]:
test_dir = "/Users/mariappan.subramanian/Library/CloudStorage/OneDrive-TheTradeDesk/MMM/BudgetOptimizer/trash"

demo_model_path = '/Users/mariappan.subramanian/Documents/repo/forked/meridian/demo/saved_models'
demo_model_file = f"{demo_model_path}/demo_model_geo_all_channels.pkl"
mmm = model.load_mmm(demo_model_file)

In [61]:
# export median values of model parameters
inference_data = mmm.inference_data['posterior']

# required parameters
media_params = [
    constants.EC_M,
    constants.SLOPE_M,
    constants.ALPHA_M,
    constants.BETA_GM,
]

rf_params = [
    constants.EC_RF,
    constants.SLOPE_RF,
    constants.ALPHA_RF,
    constants.BETA_GRF,
]


media_trans_params, rf_trans_params, media_beta_params, rf_beta_params = [], [], [], []
for param in media_params + rf_params:
  if param in inference_data:
    if param in media_params:
      if param in ('beta_gm', 'beta_grf'):
        media_beta_params.append(inference_data[param].median(dim=('chain', 'draw')).to_pandas().reset_index())
      else:
        media_trans_params.append(inference_data[param].median(dim=('chain', 'draw')).to_pandas().reset_index())
    elif param in rf_params:
      if param in ('beta_gm', 'beta_grf'):
        rf_beta_params.append(inference_data[param].median(dim=('chain', 'draw')).to_pandas().reset_index())
      else:
        rf_trans_params.append(inference_data[param].median(dim=('chain', 'draw')).to_pandas().reset_index())

# merge the results
def join_df_list_fn(df_list, join_on):
  return functools.reduce(
      lambda left, right: pd.merge(left, right, on=join_on, how='outer'),
      df_list
  )

media_trans_params_df = join_df_list_fn(media_trans_params, 'media_channel')
rf_trans_params_df = join_df_list_fn(rf_trans_params, 'rf_channel')
beta_params_df = join_df_list_fn(media_beta_params + rf_beta_params, 'geo')

# write to an excel file
with pd.ExcelWriter(f'{test_dir}/mmm_params.xlsx') as writer:
  media_trans_params_df.to_excel(writer, sheet_name='media_trans_params', index=False)
  rf_trans_params_df.to_excel(writer, sheet_name='rf_trans_params', index=False)
  beta_params_df.to_excel(writer, sheet_name='beta_params', index=False)


In [60]:
beta_params_df

,geo,Channel0,Channel1,Channel2,Channel3
0,Geo0,0.591570,0.576957,0.650515,0.542482
1,Geo1,0.605447,0.586188,0.615318,0.518652
2,Geo10,0.580106,0.553298,0.611185,0.571055
3,Geo11,0.578072,0.558315,0.620357,0.523499
4,Geo12,0.579069,0.558864,0.625085,0.547776
5,Geo13,0.572960,0.563209,0.619132,0.534187
6,Geo14,0.570950,0.552127,0.629092,0.517793
7,Geo15,0.589869,0.563430,0.647530,0.527397
8,Geo16,0.578935,0.583816,0.620215,0.525091
9,Geo17,0.570547,0.549138,0.633408,0.528098


In [58]:
rf_beta_params[0]

rf_channel,geo,Channel3
0,Geo0,0.542482
1,Geo1,0.518652
2,Geo2,0.547071
3,Geo3,0.557285
4,Geo4,0.529449
5,Geo5,0.537765
6,Geo6,0.523995
7,Geo7,0.531292
8,Geo8,0.548735
9,Geo9,0.542647


In [42]:
# Method 1: Using functools.reduce() - most efficient
merged_trans_params = functools.reduce(
    lambda left, right: pd.merge(left, right, on='media_channel', how='outer'),
    trans_params
)

merged_trans_params

KeyError: 'media_channel'

In [37]:
trans_params[0]

,media_channel,ec_m
0,Channel0,1.529526
1,Channel1,1.232294
2,Channel2,1.164748


In [38]:
trans_params[1]

,media_channel,slope_m
0,Channel0,1.0
1,Channel1,1.0
2,Channel2,1.0


In [ ]:
# Alternative Method 2: Using a loop (more readable for some)
# merged_df = trans_params[0].copy()
# for df in trans_params[1:]:
#     merged_df = pd.merge(merged_df, df, on='media_channel', how='outer')

# Alternative Method 3: If you also want to merge beta_params
if beta_params:
    merged_beta_params = functools.reduce(
        lambda left, right: pd.merge(left, right, on='geo', how='outer'),
        beta_params
    )
    print("Beta parameters merged:")
    display(merged_beta_params)
